# Experiment: CV Import Quality Benchmark

Objective:
- Measure autosuggestion precision and recall on diverse CV snippets under free-tier constraints.
- Enforce hard rules: every candidate must include evidence and final outputs must be taxonomy `skill_id` values only.

Success criteria:
- Recall >= 0.85 for expected taxonomy matches
- Precision >= 0.80 for mapped suggestions
- Evidence coverage = 1.00 (no candidate without evidence snippet)
- Invalid export IDs = 0


In [ ]:
# Setup benchmark cases and taxonomy allowlist
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import statistics

@dataclass
class BenchmarkCase:
    case_id: str
    text: str
    expected_skill_ids: set[str]
    expected_unmapped_terms: set[str]

BENCHMARK_CASES = [
    BenchmarkCase(
        case_id="frontend_core",
        text="Built React and Next.js apps in TypeScript and Node.js.",
        expected_skill_ids={"skill_react", "skill_nextjs", "skill_typescript", "skill_nodejs"},
        expected_unmapped_terms=set(),
    ),
    BenchmarkCase(
        case_id="ml_notebooks",
        text="Trained Python models in Jupyter Notebook and Google Colab.",
        expected_skill_ids={"skill_python", "skill_jupyter_notebook"},
        expected_unmapped_terms=set(),
    ),
    BenchmarkCase(
        case_id="devops_symbols",
        text="Implemented CI/CD in GitHub Actions, shipping C++ and C# services with Docker and Kubernetes.",
        expected_skill_ids={
            "skill_ci_cd", "skill_github_actions", "skill_c_plus_plus", "skill_c_sharp", "skill_docker", "skill_kubernetes"
        },
        expected_unmapped_terms=set(),
    ),
    BenchmarkCase(
        case_id="unmapped_guard",
        text="Built low-latency services in Rust.",
        expected_skill_ids=set(),
        expected_unmapped_terms={"rust"},
    ),
]

TAXONOMY_ALLOWLIST = {
    "skill_react", "skill_nextjs", "skill_typescript", "skill_nodejs", "skill_python",
    "skill_jupyter_notebook", "skill_ci_cd", "skill_github_actions", "skill_c_plus_plus",
    "skill_c_sharp", "skill_docker", "skill_kubernetes", "skill_communication", "skill_leadership",
}

print(f"Loaded {len(BENCHMARK_CASES)} benchmark cases")


## Plan
- Run the API and save responses to `tmp/cv-import-benchmark-results.json`.
- Evaluate per-case recall, precision, evidence coverage, unmapped handling, and export allowlist validity.
- Repeat with threshold changes (fuzzy, semantic, pool size) and compare metrics before changing production defaults.


In [ ]:
# Evaluation helpers
def load_results(path: str = "tmp/cv-import-benchmark-results.json") -> dict:
    payload = json.loads(Path(path).read_text())
    return payload

def evaluate(payload: dict) -> dict:
    by_id = {doc["document_id"]: doc for doc in payload["documents"]}
    case_metrics = []
    evidence_failures = 0
    invalid_export_ids = set()

    for case in BENCHMARK_CASES:
        doc = by_id.get(case.case_id, {"candidates": []})
        candidates = doc.get("candidates", [])
        predicted_ids = {s["skill_id"] for c in candidates for s in c.get("suggestions", [])}
        for skill_id in predicted_ids:
            if skill_id not in TAXONOMY_ALLOWLIST:
                invalid_export_ids.add(skill_id)

        for candidate in candidates:
            if not candidate.get("evidence_snippets"):
                evidence_failures += 1

        tp = len(predicted_ids & case.expected_skill_ids)
        fp = len(predicted_ids - case.expected_skill_ids)
        fn = len(case.expected_skill_ids - predicted_ids)
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1) if case.expected_skill_ids else 1.0

        unmapped_terms = {c["raw_skill_text"].lower() for c in candidates if c.get("unmapped_candidate")}
        unmapped_ok = case.expected_unmapped_terms.issubset(unmapped_terms)

        case_metrics.append({
            "case_id": case.case_id,
            "precision": precision,
            "recall": recall,
            "predicted_count": len(predicted_ids),
            "expected_count": len(case.expected_skill_ids),
            "unmapped_ok": unmapped_ok,
        })

    return {
        "case_metrics": case_metrics,
        "mean_precision": statistics.fmean(item["precision"] for item in case_metrics),
        "mean_recall": statistics.fmean(item["recall"] for item in case_metrics),
        "evidence_failures": evidence_failures,
        "invalid_export_ids": sorted(invalid_export_ids),
    }


## Results
- Run the evaluation cell below after each threshold or matcher change.
- Compare against prior run metrics and keep a changelog of config changes that moved precision/recall.


In [ ]:
# Example usage (expects API output JSON file at tmp/cv-import-benchmark-results.json)
# payload = load_results()
# report = evaluate(payload)
# report

# Optional fallback demo with synthetic perfect payload
synthetic_payload = {
    "documents": [
        {"document_id": "frontend_core", "candidates": [{"raw_skill_text": "React", "evidence_snippets": ["React"], "suggestions": [{"skill_id": "skill_react"}], "unmapped_candidate": False}]},
        {"document_id": "ml_notebooks", "candidates": [{"raw_skill_text": "Python", "evidence_snippets": ["Python"], "suggestions": [{"skill_id": "skill_python"}], "unmapped_candidate": False}]},
        {"document_id": "devops_symbols", "candidates": [{"raw_skill_text": "CI/CD", "evidence_snippets": ["CI/CD"], "suggestions": [{"skill_id": "skill_ci_cd"}], "unmapped_candidate": False}]},
        {"document_id": "unmapped_guard", "candidates": [{"raw_skill_text": "Rust", "evidence_snippets": ["Rust"], "suggestions": [], "unmapped_candidate": True}]},
    ]
}

demo = evaluate(synthetic_payload)
demo


## Next steps
- If deterministic metrics meet targets, keep production path model-free for Vercel Free stability.
- If metrics miss targets, run a Colab-only experiment with a tiny local model and compare against this baseline before any production rollout.
- Keep all production exports strict to taxonomy `skill_id` allowlist regardless of extraction method.
